# **Z-Image-Turbo for Text to Image**
- Github project page: https://github.com/Tongyi-MAI/Z-Image
- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

In [ ]:
# @title 💥1. Prepare Environment
import os
import sys
import subprocess
from IPython.display import clear_output

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

print("Initializing high-speed backend...")
run("pip install -q torch torchvision torchaudio")
run("apt-get -y install -qq aria2 ffmpeg")
run("pip install -q torchsde av diffusers accelerate einops spandrel ipywidgets nest_asyncio Pillow albumentations onnx opencv-python onnxruntime")

if not os.path.exists("/content/ComfyUI"):
    run("git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI")
    run("pip install -r /content/ComfyUI/requirements.txt")

clear_output()
print("✅ Environment Setup Complete!")

In [ ]:
# @title 💥2. Download Checkpoints {"single-column":true}
import os
import subprocess
from pathlib import Path

civitai_token = "" # @param {type:"string"}

def pull(url, target_dir, fallback_name):
    if not url or not url.strip(): return fallback_name
    Path(target_dir).mkdir(parents=True, exist_ok=True)
    fname = url.split('/')[-1].split('?')[0] or fallback_name
    out = os.path.join(target_dir, fname)
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print(f"✓ Present: {fname}")
        return fname
    print(f"Downloading {fname}...")
    if 'civitai.com' in url.lower():
        auth = f"{url}&token={civitai_token}" if '?' in url else f"{url}?token={civitai_token}"
        subprocess.run(f'wget --max-redirect=10 --show-progress "{auth}" -O "{out}"', shell=True)
    else:
        subprocess.run(['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', target_dir, '-o', fname, url])
    return fname

# Asset: (z_image_turbo_bf16.safetensors)
z_image_model = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors' # @param {type:"string"}
asset_1 = pull(z_image_model, "/content/ComfyUI/models/diffusion_models", 'z_image_turbo_bf16.safetensors')

# Asset: (qwen_3_4b.safetensors)
text_encoder_model = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors' # @param {type:"string"}
asset_2 = pull(text_encoder_model, "/content/ComfyUI/models/text_encoders", 'qwen_3_4b.safetensors')

# Asset: (ae.safetensors)
vae_model = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors' # @param {type:"string"}
asset_3 = pull(vae_model, "/content/ComfyUI/models/vae", 'ae.safetensors')

print("✅ Models Download Complete!")


In [ ]:
# @title 💥3. Generate Image  {"single-column":true}

import os
import random
import ipywidgets as widgets
from IPython.display import display

os.makedirs('/content/ComfyUI/input', exist_ok=True)

# @markdown #### **Prompt**
your_prompt = "A Latina female with thick wavy hair, harbor boats and pastel houses behind. Breezy seaside light, warm tones, cinematic close-up. " # @param {type:"string"}

# @markdown #### **Dimensions**
width_13 = 1024 # @param {type:"slider", min:64, max:4096, step:8}
height_13 = 1024 # @param {type:"slider", min:64, max:4096, step:8}

# @markdown #### **Sampling Parameters**
seed_3 = 0 # @param {type:"integer"}
steps_3 = 8 # @param {type:"slider", min:1, max:100, step:1}
cfg_3 = 1.0 # @param {type:"slider", min:1.0, max:30.0, step:0.5}
sampler_3 = "res_multistep" # @param ["euler", "euler_ancestral", "heun", "dpmpp_2m", "dpmpp_sde", "res_multistep"]
z_model_weight_dtype="fp8_e4m3fn" # @param ["default", "fp8_e4m3fn"]
# @markdown ##### *If using the T4, set `z_model_weight_dtype` to "fp8_e4m3fn" to avoid crashes*

if seed_3 == 0: seed_3 = random.randint(1, 2**32-1)


import sys
import gc

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from typing import Union, Sequence, Mapping, Any
from PIL import Image
from IPython.display import display, clear_output

# Initialize internal target resolution directories
sys.path.append('/content/ComfyUI')
%cd -q /content/ComfyUI

def get_value_at_index(obj: Union[Sequence, Mapping, Any], index: int) -> Any:
    """Returns the value at the given index of a sequence or mapping.

    If the object is a sequence (like list or string), returns the value at the given index.
    If the object is a mapping (like a dictionary), returns the value at the index-th key.

    Some return a dictionary, in these cases, we look for the "results" key

    Args:
        obj (Union[Sequence, Mapping, Any]): The object to retrieve the value from.
        index (int): The index of the value to retrieve.

    Returns:
        Any: The value at the given index.
    """
    try:
        return obj[index]
    except KeyError:
        if "result" in obj:
            return obj["result"][index]
        return obj["results"][index]
    except (TypeError, AttributeError):
        return obj

from comfy_extras.nodes_model_advanced import ModelSamplingAuraFlow
from comfy_extras.nodes_sd3 import EmptySD3LatentImage
from nodes import CLIPLoader
from nodes import CLIPTextEncode
from nodes import ConditioningZeroOut
from nodes import KSampler
from nodes import SaveImage
from nodes import UNETLoader
from nodes import VAEDecode
from nodes import VAELoader


clip_loader = CLIPLoader()
vae_loader = VAELoader()
unet_loader = UNETLoader()
clip_text_encode = CLIPTextEncode()
empty_sd3_latent_image = EmptySD3LatentImage()
aura_flow = ModelSamplingAuraFlow()
conditioning_zero_out = ConditioningZeroOut()
k_sampler = KSampler()
vae_decode = VAEDecode()
save_image = SaveImage()


print("Loading text encoder...")

clip = clip_loader.load_clip(
    clip_name="qwen_3_4b.safetensors",
    type="lumina2",
    device="default"
)

conditioning = clip_text_encode.encode(
    text=your_prompt,
    clip=get_value_at_index(clip, 0)
)[0]


try:
    del clip
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()


latent_image = empty_sd3_latent_image.execute(
    width=width_13,
    height=height_13,
    batch_size=1
)
clear_output()
print("Loading z_image_turbo checkpoint...")

model = unet_loader.load_unet(
    unet_name="z_image_turbo_bf16.safetensors",
    weight_dtype=z_model_weight_dtype
)

model_sampling = aura_flow.patch_aura(
    model=get_value_at_index(model, 0),
    shift=3
)

conditioning_zero_out = conditioning_zero_out.zero_out(conditioning=conditioning)

clear_output()
print("Generating Image...")

samples = k_sampler.sample(
    model=get_value_at_index(model_sampling, 0),
    seed=seed_3,
    steps=steps_3,
    cfg=cfg_3,
    sampler_name=sampler_3,
    scheduler="simple",
    positive=conditioning,
    negative=get_value_at_index(conditioning_zero_out, 0),
    latent_image=get_value_at_index(latent_image, 0),
    denoise=1
)

try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

vae = vae_loader.load_vae(vae_name="ae.safetensors")

images = vae_decode.decode(
    samples=get_value_at_index(samples, 0),
    vae=get_value_at_index(vae, 0)
)

try:
    del vae
    del conditioning
    del conditioning_zero_out
    del latent_image
    del model_sampling
    del samples

except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

save_image.save_images(
    images=get_value_at_index(images, 0),
    filename_prefix="z-image-turbo"
)

clear_output()

# 3. Display Generated Media
output_dir = '/content/ComfyUI/output'
if os.path.exists(output_dir):
    images = [os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    if images:
        latest = max(images, key=os.path.getmtime)
        print(f'Displaying generated media: {os.path.basename(latest)}')
        display(Image.open(latest))
    else: print('Execution complete. No standard output images dynamically registered.')
else: print('Could not find image path.')
